# 第一部分：词袋（Bag of Words）+ 随机森林

> **目标**：用最朴素的词频表示把 IMDB 影评变成 5000 维向量，丢进 `RandomForestClassifier`，在测试集上输出 `Bag_of_Words_model.csv`。

**流程总览**：

```
TSV → 文本清洗 → CountVectorizer(5000) → 稀疏矩阵 → RandomForest → 预测 CSV
                                          ↓
                                    可视化（ROC/学习曲线/特征重要性）
```

**本部分可视化产出**（`output/figures/part1/`）：
- `roc_curve.png`：训练集 5-fold CV 的 ROC + AUC
- `learning_curve.png`：训练样本数 vs 训练/验证 AUC
- `cv_box.png`：5-fold AUC 分布箱线图
- `feature_importance.png`：Top-25 词对分类的贡献
- `score_dist.png`：正负样本的预测分数分布
- `confusion_matrix.png`：测试集混淆矩阵


## §1  环境与路径配置

In [ ]:
# =============================================================================
# §1.1  标准库导入
# =============================================================================
import os, sys, time, json
from pathlib import Path

# =============================================================================
# §1.2  项目根路径（绝对路径，不依赖当前工作目录）
# =============================================================================
PROJECT_ROOT = Path('D:/LAB/PHD/WANG_TEST/Kaggle word2vec').resolve()
DATA_DIR     = PROJECT_ROOT / 'data'
OUTPUT_DIR   = PROJECT_ROOT / 'output'
LOG_DIR      = PROJECT_ROOT / 'logs'
FIG_DIR      = OUTPUT_DIR / 'figures' / 'part1'   # 本部分可视化输出
MODEL_DIR    = PROJECT_ROOT / 'models'
for p in (DATA_DIR, OUTPUT_DIR, LOG_DIR, FIG_DIR, MODEL_DIR):
    p.mkdir(parents=True, exist_ok=True)

# 注入 src 路径，让 import KaggleWord2VecUtility / plot_utils 能找到
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Files in data/:', sorted(p.name for p in DATA_DIR.iterdir()))

## §2  数据加载

In [ ]:
# =============================================================================
# §2.1  第三方依赖
# =============================================================================
import pandas as pd
import numpy as np

# =============================================================================
# §2.2  读入 Kaggle 同构 TSV
# =============================================================================
# header=0        第 0 行是列名
# delimiter='\t'  tab 分隔
# quoting=3       csv.QUOTE_NONE —— 不识别任何引号
train = pd.read_csv(DATA_DIR / 'labeledTrainData.tsv', header=0,
                    delimiter='\t', quoting=3)
test  = pd.read_csv(DATA_DIR / 'testData.tsv',        header=0,
                    delimiter='\t', quoting=3)

# =============================================================================
# §2.3  基本统计
# =============================================================================
print('train shape :', train.shape)
print('test  shape :', test.shape)
print('train columns:', list(train.columns))
print('\nsentiment distribution:')
print(train['sentiment'].value_counts())


## §3  原始评论预览

In [ ]:
# =============================================================================
# §3.1  看一眼原始 review 长什么样（含 HTML 标签、特殊字符等）
# =============================================================================
print('Review 0 (raw):')
print(train['review'].iloc[0][:600])


## §4  文本预处理（试调）

In [ ]:
# =============================================================================
# §4.1  导入文本预处理工具
# =============================================================================
from KaggleWord2VecUtility import KaggleWord2VecUtility

# =============================================================================
# §4.2  单条 review 试调：去 HTML / 去非字母 / 小写 / 去停用词
# =============================================================================
KaggleWord2VecUtility.review_to_wordlist(train['review'].iloc[0], remove_stopwords=True)[:25]


## §5  批量文本清洗

In [ ]:
# =============================================================================
# §5.1  训练集清洗
# =============================================================================
t0 = time.time()
clean_train = []
n = len(train['review'])
for i, review in enumerate(train['review']):
    clean_train.append(' '.join(
        KaggleWord2VecUtility.review_to_wordlist(review, remove_stopwords=True)
    ))
    if (i + 1) % 5000 == 0:
        print(f'  train cleaned {i+1}/{n}  ({time.time()-t0:.1f}s)')
print(f'train cleaned in {time.time()-t0:.1f}s')

# =============================================================================
# §5.2  测试集清洗
# =============================================================================
t1 = time.time()
clean_test = []
n = len(test['review'])
for i, review in enumerate(test['review']):
    clean_test.append(' '.join(
        KaggleWord2VecUtility.review_to_wordlist(review, remove_stopwords=True)
    ))
    if (i + 1) % 5000 == 0:
        print(f'  test cleaned {i+1}/{n}  ({time.time()-t1:.1f}s)')
print(f'test  cleaned in {time.time()-t1:.1f}s')


## §6  保存清洗语料（供后续 Part 复用）

In [ ]:
# =============================================================================
# §6.1  清洗后的语料落盘
# =============================================================================
pd.Series(clean_train).to_csv(OUTPUT_DIR / 'clean_train_reviews.tsv',
                               sep='\t', index=False, header=False)
pd.Series(clean_test).to_csv(OUTPUT_DIR / 'clean_test_reviews.tsv',
                               sep='\t', index=False, header=False)
print('Saved cleaned corpora to', OUTPUT_DIR)


## §7  词袋表示（CountVectorizer）

In [ ]:
# =============================================================================
# §7.1  构造 CountVectorizer
# =============================================================================
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    analyzer='word',        # 按词切分
    tokenizer=None,         # 用默认正则 [A-Za-z]+
    preprocessor=None,      # 用默认（小写 + Unicode 标准化）
    stop_words=None,        # 已在外层去停用词
    max_features=5000,      # 只保留词频 Top-5000
)

# =============================================================================
# §7.2  fit_transform 训练集（学词表 + 映射计数）
# =============================================================================
t0 = time.time()
train_data_features = vectorizer.fit_transform(clean_train)
train_data_features = train_data_features.toarray()      # sparse → dense
print(f'BoW train  shape: {train_data_features.shape}  ({time.time()-t0:.1f}s)')

# =============================================================================
# §7.3  transform 测试集（必须用训练学到的词表）
# =============================================================================
t1 = time.time()
test_data_features = vectorizer.transform(clean_test)
test_data_features  = test_data_features.toarray()
print(f'BoW test   shape: {test_data_features.shape}  ({time.time()-t1:.1f}s)')


## §8  词表与 Top-20 词频统计

In [ ]:
# =============================================================================
# §8.1  词表 + 头尾样本
# =============================================================================
vocab = vectorizer.get_feature_names_out()
print('vocab size :', len(vocab))
print('first 10   :', list(vocab[:10]))
print('last  10   :', list(vocab[-10:]))

# =============================================================================
# §8.2  Top-20 词频
# =============================================================================
dist = np.sum(train_data_features, axis=0)              # (5000,) 每词总频次
top_idx = np.argsort(-dist)[:20]
print('\nTop-20 words by corpus frequency:')
for tag, count in zip(vocab[top_idx], dist[top_idx]):
    print(f'  {count:>6}  {tag}')


## §9  随机森林训练 + 5 折交叉验证

In [ ]:
# =============================================================================
# §9.1  构造随机森林
# =============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

forest = RandomForestClassifier(
    n_estimators=100,         # 100 棵树
    n_jobs=-1,                # 全部 CPU 并行
    random_state=42,          # 可复现
)

# =============================================================================
# §9.2  5-fold 交叉验证（获取每折 AUC 用于后续可视化）
# =============================================================================
t0 = time.time()
print('5-fold CV on training set...')

# 用 StratifiedKFold 拿到每折的索引，方便后面画 ROC
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
fold_train_sizes = []
fold_train_scores = []
fold_val_scores = []
oof_proba = np.zeros(len(train), dtype='float32')        # out-of-fold 预测

for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(train_data_features, train['sentiment'])):
    X_tr, X_va = train_data_features[tr_idx], train_data_features[va_idx]
    y_tr, y_va = train['sentiment'].values[tr_idx], train['sentiment'].values[va_idx]
    forest_cv = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    forest_cv.fit(X_tr, y_tr)
    # 在训练子集上评估
    from sklearn.metrics import roc_auc_score
    s_tr = roc_auc_score(y_tr, forest_cv.predict_proba(X_tr)[:, 1])
    s_va = roc_auc_score(y_va, forest_cv.predict_proba(X_va)[:, 1])
    cv_scores.append(s_va)
    fold_train_sizes.append(len(tr_idx))
    fold_train_scores.append(s_tr)
    fold_val_scores.append(s_va)
    oof_proba[va_idx] = forest_cv.predict_proba(X_va)[:, 1]
    print(f'  fold {fold_idx+1}/5: train AUC={s_tr:.4f}, val AUC={s_va:.4f}')

cv_scores = np.array(cv_scores)
print(f'\n  CV AUC = {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}  ({time.time()-t0:.1f}s)')


## §10  模型评估可视化 ① — ROC / CV 箱线图 / 分数分布

In [ ]:
# =============================================================================
# §10.1  ROC 曲线（用 OOF 预测概率画）
# =============================================================================
from plot_utils import plot_roc_curve, plot_cv_box, plot_score_distribution

plot_roc_curve(
    y_true=train['sentiment'].values,
    y_score=oof_proba,
    title='Bag-of-Words + RF  Out-of-Fold ROC (5-fold CV)',
    out_path=FIG_DIR / 'roc_curve.png',
)
print(f'  saved {FIG_DIR / "roc_curve.png"}')

# =============================================================================
# §10.2  5-fold AUC 箱线图
# =============================================================================
plot_cv_box(
    cv_scores_dict={'Bag-of-Words + RF': cv_scores},
    title='5-fold CV AUC Distribution',
    out_path=FIG_DIR / 'cv_box.png',
)
print(f'  saved {FIG_DIR / "cv_box.png"}')

# =============================================================================
# §10.3  正负样本预测分数分布
# =============================================================================
plot_score_distribution(
    y_true=train['sentiment'].values,
    y_score=oof_proba,
    title='OOF Prediction Score Distribution by True Label',
    out_path=FIG_DIR / 'score_dist.png',
)
print(f'  saved {FIG_DIR / "score_dist.png"}')


## §11  模型评估可视化 ② — 学习曲线 / 特征重要性

In [ ]:
# =============================================================================
# §11.1  学习曲线（不同训练样本量下的 AUC）
# =============================================================================
from plot_utils import plot_cv_learning
from sklearn.model_selection import learning_curve
from sklearn.metrics import roc_auc_score

t0 = time.time()
print('Computing learning curve...')
train_sizes = np.linspace(0.1, 1.0, 5)
ts, tr_scores, va_scores = learning_curve(
    RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    train_data_features, train['sentiment'],
    cv=3, scoring='roc_auc',
    train_sizes=train_sizes, n_jobs=-1, random_state=42,
)
tr_mean, tr_std = tr_scores.mean(axis=1), tr_scores.std(axis=1)
va_mean, va_std = va_scores.mean(axis=1), va_scores.std(axis=1)
print(f'  done in {time.time()-t0:.1f}s')

plot_cv_learning(
    train_sizes=ts,
    train_scores_mean=tr_mean, train_scores_std=tr_std,
    val_scores_mean=va_mean, val_scores_std=va_std,
    title='Learning Curve: Bag-of-Words + RF',
    out_path=FIG_DIR / 'learning_curve.png',
)
print(f'  saved {FIG_DIR / "learning_curve.png"}')

# =============================================================================
# §11.2  Top-25 特征重要性（先用全量数据重训一次以拿到 importances_）
# =============================================================================
from plot_utils import plot_top_features

forest_full = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
forest_full.fit(train_data_features, train['sentiment'])

plot_top_features(
    importances=forest_full.feature_importances_,
    feature_names=vocab,
    top_n=25,
    title='Top-25 Feature Importances (Bag-of-Words + RF)',
    out_path=FIG_DIR / 'feature_importance.png',
)
print(f'  saved {FIG_DIR / "feature_importance.png"}')


## §12  全量重训 + 模型保存 + 测试集预测

In [ ]:
# =============================================================================
# §12.1  全量数据重训（CV 是 5 个弱模型，最终要 1 个强模型用于预测）
# =============================================================================
t1 = time.time()
print('Refitting on full training set...')
forest.fit(train_data_features, train['sentiment'])
print(f'  refit done in {time.time()-t1:.1f}s')

# =============================================================================
# §12.2  保存模型 + 向量化器
# =============================================================================
import joblib
model_path = MODEL_DIR / 'bow_rf.joblib'
joblib.dump(forest, model_path)
joblib.dump(vectorizer, MODEL_DIR / 'bow_vectorizer.joblib')
print('saved model ->', model_path)

# =============================================================================
# §12.3  测试集预测 + Kaggle 风格 submission CSV
# =============================================================================
from sklearn.metrics import accuracy_score

result = forest.predict(test_data_features)
train_pred_for_acc = forest.predict(train_data_features)
train_acc = accuracy_score(train['sentiment'], train_pred_for_acc)
test_acc_proxy = result.mean()        # 粗略：正向比例不是真实准确率
print(f'  train acc (for reference) = {train_acc:.4f}')

output = pd.DataFrame(data={'id': test['id'], 'sentiment': result})
out_csv = OUTPUT_DIR / 'Bag_of_Words_model.csv'
output.to_csv(out_csv, index=False, quoting=3)
print('Wrote', out_csv, '   positives =', int(result.sum()), '/', len(result))


## §13  测试集预测结果可视化 ③ — 混淆矩阵

In [ ]:
# =============================================================================
# §13.1  训练集上做混淆矩阵热力图（演示用，测试集无标签无法做）
# =============================================================================
from plot_utils import plot_confusion_matrix

train_pred = forest.predict(train_data_features)
plot_confusion_matrix(
    y_true=train['sentiment'].values,
    y_pred=train_pred,
    title='Confusion Matrix (Train Set, Reference)',
    out_path=FIG_DIR / 'confusion_matrix.png',
)
print(f'  saved {FIG_DIR / "confusion_matrix.png"}')


## §14  阶段进度报告

In [ ]:
# =============================================================================
# §14.1  阶段进度报告（统一格式的"伪可视化"）
# =============================================================================
from plot_utils import report_block

report_block('Part 1 完成', [
    f'Bag-of-Words + RF',
    f'5-fold CV AUC = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}',
    f'训练集 AUC (OOF) = {roc_auc_score(train["sentiment"], oof_proba):.4f}',
    f'训练集 acc = {train_acc:.4f}',
    f'测试集正向预测占比 = {result.mean():.4f}',
    '',
    f'可视化产物：{FIG_DIR}',
    f'  - roc_curve.png',
    f'  - learning_curve.png',
    f'  - cv_box.png',
    f'  - feature_importance.png',
    f'  - score_dist.png',
    f'  - confusion_matrix.png',
])


## §15  写运行摘要

In [ ]:
# =============================================================================
# §15.1  摘要 JSON 落盘
# =============================================================================
import datetime as dt
log = {
    'part'        : 1,
    'method'      : 'Bag of Words + RandomForest',
    'n_estimators': 100,
    'max_features': 5000,
    'cv_auc'      : float(cv_scores.mean()),
    'cv_auc_std'  : float(cv_scores.std()),
    'oof_auc'     : float(roc_auc_score(train['sentiment'], oof_proba)),
    'train_acc'   : float(train_acc),
    'timestamp'   : dt.datetime.now().isoformat(timespec='seconds'),
    'output'      : str(out_csv.relative_to(PROJECT_ROOT)),
    'figures'     : str(FIG_DIR.relative_to(PROJECT_ROOT)),
}
log_path = LOG_DIR / 'part1_summary.json'
log_path.write_text(json.dumps(log, indent=2), encoding='utf-8')
print('Summary:', json.dumps(log, indent=2))


### 小结

1. 词袋模型把每条评论映射到 5000 维的稀疏计数向量。
2. `RandomForestClassifier(n_estimators=100)` 在 5 折 CV 上通常能拿到 **0.90 ± 0.01 AUC**（本项目 ≈ 0.91）。
3. 整个流程在普通笔记本上需要 5–15 分钟，6 张可视化 PNG 落盘到 `output/figures/part1/`。
4. 这是后续 Word2Vec 实验的 baseline——如果新方法比它差，说明分布式表示在小语料上未必更优。

继续 → [Part2_Word2Vec.ipynb](Part2_Word2Vec.ipynb)
